# Tutorial 1: Working with ENDF (Evaluated Nuclear Data File) Data

## Overview

ENDF is the primary evaluated nuclear data library used in the United States. It contains comprehensive, validated nuclear data that has been carefully evaluated by experts.

### What you'll learn:
- How to download **REAL** ENDF data from JANIS
- How to load and visualize actual cross-sections
- Understanding different reaction types (MT numbers)
- Working with real nuclear data for ML applications

### Prerequisites:
```bash
pip install matplotlib numpy pandas
```

### ⚠️ IMPORTANT: This Tutorial Uses REAL DATA ONLY

**You MUST download actual nuclear data from JANIS before running this tutorial.**

No simulated or synthetic data is used. All analysis is performed on real, evaluated nuclear data from official sources.

## 1. Download Real ENDF Data from JANIS

### Step-by-Step Instructions:

#### **Step 1:** Open JANIS Website
Go to: **https://www.oecd-nea.org/janisweb/**

#### **Step 2:** Navigate to Cross Sections
- Click **"Search"** in the top menu
- Select **"Cross Sections"**

#### **Step 3:** Configure Search for U-235 Fission
```
Projectile:  n (neutron)
Target:      U-235
Reaction:    (n,f)   [fission]
Library:     ✓ ENDF/B-VIII.0
```

#### **Step 4:** Plot and Export
1. Click **"Plot Data"**
2. Wait for plot to load
3. Click **"Export" or "Download"** button
4. Select **"CSV"** format
5. Save as: `u235_fission_endf8.csv`

#### **Step 5:** Place File in Correct Location
Save the downloaded CSV to:
```
../data/endf/u235_fission_endf8.csv
```

### Download Additional Reactions:

Repeat the above process for:

**Capture (n,γ):**
- Reaction: `(n,g)`
- Save as: `u235_capture_endf8.csv`

**Elastic Scattering:**
- Reaction: `(n,el)` or `(n,n)`  
- Save as: `u235_elastic_endf8.csv`

---

### ✅ Checklist Before Proceeding:
- [ ] Downloaded `u235_fission_endf8.csv`
- [ ] Downloaded `u235_capture_endf8.csv`
- [ ] Downloaded `u235_elastic_endf8.csv`
- [ ] All files in `../data/endf/` directory

**Only proceed once you have downloaded the real data!**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Create data directory
data_dir = Path('../data/endf')
data_dir.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete!")
print(f"Data directory: {data_dir.absolute()}")
print("\n" + "="*70)
print("IMPORTANT: You must download real data from JANIS before proceeding!")
print("See instructions above for download steps.")
print("="*70)

## 2. Load Real ENDF Data

Now we'll load the **real** ENDF data you downloaded from JANIS.

In [ ]:
# Define expected file paths
fission_file = data_dir / 'u235_fission_endf8.csv'
capture_file = data_dir / 'u235_capture_endf8.csv'
elastic_file = data_dir / 'u235_elastic_endf8.csv'

# Check if fission data exists (required)
if not fission_file.exists():
    print("❌ ERROR: Real data file not found!")
    print(f"\nExpected file: {fission_file}")
    print("\n📥 PLEASE DOWNLOAD FROM JANIS:")
    print("   1. Go to https://www.oecd-nea.org/janisweb/")
    print("   2. Search → Cross Sections")
    print("   3. Select: U-235, (n,f), ENDF/B-VIII.0")
    print("   4. Plot Data → Export → CSV")
    print(f"   5. Save as: {fission_file.name}")
    print(f"   6. Place in: {data_dir.absolute()}")
    print("\n⚠️  This tutorial requires REAL data. Cannot proceed without it.")
    raise FileNotFoundError(f"Required data file not found: {fission_file}")

# Load fission data
print(f"✓ Loading real ENDF data from: {fission_file.name}")
df_fission = pd.read_csv(fission_file)

# Display info about the real data
print(f"\n✓ Successfully loaded REAL ENDF/B-VIII.0 data!")
print(f"  Reaction: U-235(n,f) Fission")
print(f"  Data points: {len(df_fission)}")
print(f"  Columns: {list(df_fission.columns)}")
print("\nFirst few rows of REAL data:")
print(df_fission.head())

# Identify energy and cross-section columns
# JANIS CSV format may vary - adapt column names
print("\n📊 Identifying data columns...")
print(f"Available columns: {list(df_fission.columns)}")
print("\nℹ️  Note: Column names may vary. Adjust 'Energy' and 'XS' column names as needed.")

## 3. Understanding ENDF Cross-Section Data

### MT Numbers (Reaction Types):

| MT  | Reaction      | Description                  |
|-----|---------------|------------------------------|
| 1   | Total         | Total cross-section          |
| 2   | Elastic       | Elastic scattering (n,n)     |
| 4   | Inelastic     | Inelastic scattering (n,n')  |
| 18  | **(n,f)**     | **Fission**                  |
| 102 | **(n,γ)**     | **Radiative capture**        |
| 103 | (n,p)         | Proton production            |
| 107 | (n,α)         | Alpha production             |

### Energy Regions:
- **Thermal** (< 1 eV): 1/v behavior, high cross-sections
- **Epithermal** (1 eV - 100 eV): Transition region
- **Resonance** (100 eV - 10 keV): Complex resonance structure
- **Fast** (> 10 keV): Smoothly varying cross-sections

## 4. Prepare Real Data for Visualization

**Note:** JANIS CSV column names may vary. Adjust the column names below to match your downloaded file.

In [ ]:
# Standardize column names
# Adjust these based on your actual JANIS CSV column names
# Common possibilities: 'Energy', 'E', 'Energy(eV)', 'XS', 'CrossSection', 'Sigma'

# Example: If your columns are named differently, rename them:
# df_fission = df_fission.rename(columns={'E': 'Energy_eV', 'Sigma': 'CrossSection_barns'})

# For this example, assume columns are: 'Energy' and 'CrossSection'
if 'Energy_eV' not in df_fission.columns:
    # Try to find energy column
    energy_cols = [col for col in df_fission.columns if 'energy' in col.lower() or 'e' == col.lower()]
    if energy_cols:
        df_fission = df_fission.rename(columns={energy_cols[0]: 'Energy_eV'})
        print(f"✓ Renamed '{energy_cols[0]}' → 'Energy_eV'")

if 'CrossSection_barns' not in df_fission.columns:
    # Try to find cross-section column
    xs_cols = [col for col in df_fission.columns 
               if 'cross' in col.lower() or 'xs' in col.lower() or 'sigma' in col.lower()]
    if xs_cols:
        df_fission = df_fission.rename(columns={xs_cols[0]: 'CrossSection_barns'})
        print(f"✓ Renamed '{xs_cols[0]}' → 'CrossSection_barns'")

# Verify we have the required columns
required_cols = ['Energy_eV', 'CrossSection_barns']
missing_cols = [col for col in required_cols if col not in df_fission.columns]

if missing_cols:
    print(f"\n⚠️  Missing columns: {missing_cols}")
    print(f"Available columns: {list(df_fission.columns)}")
    print("\nPlease manually rename columns or adjust the code above.")
else:
    print(f"\n✓ Data ready for visualization!")
    print(f"  Energy range: {df_fission['Energy_eV'].min():.2e} - {df_fission['Energy_eV'].max():.2e} eV")
    print(f"  XS range: {df_fission['CrossSection_barns'].min():.2f} - {df_fission['CrossSection_barns'].max():.2f} barns")

## 5. Visualize Real U-235 Fission Cross-Section

In [ ]:
plt.figure(figsize=(14, 8))

plt.loglog(df_fission['Energy_eV'], df_fission['CrossSection_barns'], 
           linewidth=2, color='blue', label='U-235 (n,f) Fission [REAL DATA]')

# Mark thermal point if in data range
thermal_energy = 0.0253  # eV
if df_fission['Energy_eV'].min() <= thermal_energy <= df_fission['Energy_eV'].max():
    thermal_idx = np.argmin(np.abs(df_fission['Energy_eV'] - thermal_energy))
    thermal_xs = df_fission.iloc[thermal_idx]['CrossSection_barns']
    plt.plot(thermal_energy, thermal_xs, 'ro', markersize=10, 
             label=f'Thermal point (0.0253 eV): {thermal_xs:.1f} barns')

plt.xlabel('Neutron Energy (eV)', fontsize=14, fontweight='bold')
plt.ylabel('Fission Cross-section (barns)', fontsize=14, fontweight='bold')
plt.title('U-235 Fission Cross-Section - REAL ENDF/B-VIII.0 Data from JANIS\nMT=18: (n,f) Reaction', 
          fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='best')
plt.grid(True, alpha=0.3, which='both', linestyle='--')

plt.tight_layout()
plt.savefig(data_dir / 'u235_fission_REAL_DATA.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Plot saved: {data_dir / 'u235_fission_REAL_DATA.png'}")
print(f"\n📊 This plot uses REAL evaluated nuclear data from ENDF/B-VIII.0")
print(f"   Source: OECD NEA JANIS Database")
print(f"   Data points: {len(df_fission)}")

## 6. Load Additional Real Reactions

If you downloaded capture and elastic data, load them here:

In [ ]:
# Check and load capture data
if capture_file.exists():
    print(f"✓ Loading real capture data: {capture_file.name}")
    df_capture = pd.read_csv(capture_file)
    # Standardize column names (same process as above)
    print(f"  Capture data points: {len(df_capture)}")
else:
    print(f"ℹ️  Capture data not found: {capture_file.name}")
    print(f"   Download from JANIS with reaction (n,g)")
    df_capture = None

# Check and load elastic data
if elastic_file.exists():
    print(f"✓ Loading real elastic data: {elastic_file.name}")
    df_elastic = pd.read_csv(elastic_file)
    print(f"  Elastic data points: {len(df_elastic)}")
else:
    print(f"ℹ️  Elastic data not found: {elastic_file.name}")
    print(f"   Download from JANIS with reaction (n,el)")
    df_elastic = None

print("\n" + "="*70)
if df_capture is not None and df_elastic is not None:
    print("✓ All reaction data loaded! Ready for multi-reaction analysis.")
else:
    print("⚠️  Some reactions not loaded. Multi-reaction plots will be skipped.")
    print("   Download additional reactions from JANIS to enable full analysis.")
print("="*70)

## 7. Multi-Reaction Comparison (if all data available)

In [ ]:
if df_capture is not None and df_elastic is not None:
    # Prepare multi-reaction DataFrame
    # Note: May need to interpolate to common energy grid
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 12))
    
    # Top panel: All cross-sections
    axes[0].loglog(df_fission['Energy_eV'], df_fission['CrossSection_barns'], 
                   label='Fission (MT=18) [REAL DATA]', linewidth=2.5, color='red')
    
    # Adjust column names for capture and elastic as needed
    axes[0].loglog(df_capture.iloc[:, 0], df_capture.iloc[:, 1], 
                   label='Capture (MT=102) [REAL DATA]', linewidth=2.5, color='blue')
    axes[0].loglog(df_elastic.iloc[:, 0], df_elastic.iloc[:, 1], 
                   label='Elastic (MT=2) [REAL DATA]', linewidth=2.5, color='green')
    
    axes[0].set_ylabel('Cross-section (barns)', fontsize=14, fontweight='bold')
    axes[0].set_title('U-235 Neutron Cross-Sections - REAL ENDF/B-VIII.0 Data', 
                      fontsize=16, fontweight='bold')
    axes[0].legend(fontsize=11, loc='upper right')
    axes[0].grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.savefig(data_dir / 'u235_all_reactions_REAL_DATA.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Multi-reaction plot saved using REAL DATA")
else:
    print("⚠️  Skipping multi-reaction plot - not all data files available")
    print("   Download capture and elastic data from JANIS to enable this section")

## 8. Prepare Real Data for Machine Learning

In [ ]:
# Add ML-friendly features to real data
df_ml = df_fission.copy()

# Log-scale energy
df_ml['Log10_Energy'] = np.log10(df_ml['Energy_eV'])

# Energy region labels
df_ml['Energy_Region'] = pd.cut(df_ml['Energy_eV'], 
                                 bins=[0, 1, 100, 10000, 1e7],
                                 labels=['Thermal', 'Epithermal', 'Resonance', 'Fast'])

# Save REAL data for ML
ml_file = data_dir / 'u235_endf_REAL_ml_dataset.csv'
df_ml.to_csv(ml_file, index=False)

print(f"✓ REAL DATA ML dataset saved: {ml_file}")
print(f"\nDataset info:")
print(f"  Shape: {df_ml.shape}")
print(f"  Source: REAL ENDF/B-VIII.0 from JANIS")
print(f"  Columns: {list(df_ml.columns)}")
print("\nFirst few rows of REAL DATA:")
print(df_ml.head())

print("\n📊 Summary by energy region (REAL DATA):")
print(df_ml.groupby('Energy_Region')['CrossSection_barns'].describe())

## 9. Key Physical Insights from Real Data

### U-235 as a Fissile Material:

1. **High thermal fission XS** (~584 barns) → Fissions easily with slow neutrons
2. **Can sustain chain reaction** → η > 2 at thermal energies
3. **Resonance structure** → Self-shielding effects in reactors
4. **Fast fission** → XS ~1-2 barns at MeV energies

### What Makes This Data Valuable:
- ✅ **Evaluated by experts** - Not raw measurements
- ✅ **Validated against experiments** - Best available data
- ✅ **Used worldwide** - Standard for reactor calculations
- ✅ **Ready for ML** - Clean, validated, comprehensive

## 10. Summary

### What You Accomplished:
✅ Downloaded REAL nuclear data from official sources  
✅ Loaded and visualized actual ENDF/B-VIII.0 cross-sections  
✅ Analyzed real evaluated nuclear data  
✅ Prepared real data for machine learning applications  

### Key Takeaways:
1. **Real data is accessible** - JANIS makes it easy
2. **No simulations needed** - Work with actual evaluations
3. **Data is validated** - ENDF represents best available knowledge
4. **Ready for ML** - Clean CSV format, perfect for analysis

### Next Steps:
1. Download data for other isotopes (U-238, Pu-239, etc.)
2. Compare with experimental data (Tutorial 2: EXFOR)
3. Compare different libraries (Tutorial 4: JANIS multi-library)
4. Build ML models with this real data

## Resources

### Data Sources:
- **JANIS:** https://www.oecd-nea.org/janisweb/ (Used in this tutorial)
- **NNDC ENDF:** https://www.nndc.bnl.gov/endf/
- **IAEA NDS:** https://www-nds.iaea.org/

### Documentation:
- ENDF Format: https://www.nndc.bnl.gov/endfdocs/
- JANIS Help: https://www.oecd-nea.org/janisweb/help
- Nuclear Data: https://www-nds.iaea.org/